# Identifying Known Lenses
We have done a big major run identifying ~1,300 lenses across 250M sources. Now, need to find which are in my 50M selection for the benchmarking we want to do.

## Imports

In [11]:
import pandas as pd
import glob

import duckdb

## Getting All Found Lenses

In [3]:
csv_files = glob.glob('/media/team_workspaces/AnomalyMatch-IDR1-Search/dr1_test_run/anomaly-results/lenses/top*_000_*_000_classified.csv')
csv_files.append('/media/team_workspaces/AnomalyMatch-IDR1-Search/dr1_test_run/anomaly-results/lenses/top1_000_classified.csv')

In [4]:
csv_files

['/media/team_workspaces/AnomalyMatch-IDR1-Search/dr1_test_run/anomaly-results/lenses/top1_000_2_000_classified.csv',
 '/media/team_workspaces/AnomalyMatch-IDR1-Search/dr1_test_run/anomaly-results/lenses/top2_000_3_000_classified.csv',
 '/media/team_workspaces/AnomalyMatch-IDR1-Search/dr1_test_run/anomaly-results/lenses/top3_000_4_000_classified.csv',
 '/media/team_workspaces/AnomalyMatch-IDR1-Search/dr1_test_run/anomaly-results/lenses/top4_000_5_000_classified.csv',
 '/media/team_workspaces/AnomalyMatch-IDR1-Search/dr1_test_run/anomaly-results/lenses/top1_000_classified.csv']

In [5]:
df = pd.DataFrame(data = [], columns = ['id', 'label'])

In [6]:
for file in csv_files:
    df_tmp = pd.read_csv(file, index_col = 0)
    df = pd.concat([df, df_tmp])

In [7]:
df.label.value_counts()

label
normal     3694
anomaly    1306
Name: count, dtype: int64

In [10]:
df_anomalies = df.query('label == "anomaly"')

## Matching to Selection
Now, we can use the trick taught by Claudius to use duckdb to quickly match sources.

In [14]:
source_ids = list(df_anomalies.id)

In [15]:
con = duckdb.connect()

# Register the ID list as a table (preserves int64)
con.execute("CREATE TEMP TABLE targets AS SELECT * FROM (VALUES " +
            ",".join(f"({sid})" for sid in source_ids) +
            ") AS t(SourceID)")

result = con.execute("""
    SELECT p.SourceID, p.RA, p.Dec, p.diameter_pixel, p.fits_file_paths
    FROM read_parquet('/media/team_workspaces/AnomalyMatch-IDR1-Search/benchmarking_tests/data/source_cats/*.parquet') p
    SEMI JOIN targets t USING (SourceID)
""").df()

# result.to_parquet('matched_sources.parquet', index=False)

In [17]:
len(result)

464

Ok, this is not half a bad benchmark. We've got 464 lenses in the sample of 50M sources. Good approximation of the problem we'll be facing. Note, probably need a test run just to check there's not loads extra we've missed and we're going to pick up?

## Saving
Saving the known lenses that are in the benchmarking dataset.

In [18]:
df.to_csv('/media/team_workspaces/AnomalyMatch-IDR1-Search/benchmarking_tests/data/known_lenses/matched_known_lenses.csv')